In [1]:
import os 
import pandas as pd 
os.chdir("../FamPredAI/")
%reload_ext autoreload
%autoreload 2

In [2]:
from forecasting import forecast_from_file
from utilities import performances_forecasts, rmse
import plotly.express as px 
import plotly.graph_objects as go
import numpy as np 

In [4]:
res = []
country = []
model = []
for c in ['Nigeria','Mali', 'Yemen', 'Syria']:
  
    old = performances_forecasts(country=c, path='old_predictions/RC')    
    old2 = performances_forecasts(country=c, path='old_predictions/RC_J_code_new')
    
    path = 'old_predictions/RCNew/'
    new = []
    
    for file in [f for f in os.listdir(path) if c in f]:
        df =pd.read_csv(path+file)
        new.append(df.groupby('adm1_code').apply(lambda d: rmse(d.prediction, d.data)).median())
    
    print(np.median(new), np.median(old))
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.arange(0, len(old)), y=old, name='Old Code Old Data'))
    fig.add_trace(go.Scatter(x=np.arange(0, len(old)), y=old2, name='Old Code New Data'))
    fig.add_trace(go.Scatter(x=np.arange(0, len(old)), y=new, name='New Code'))
    fig.update_layout(yaxis=dict(title='RMSE'), xaxis=dict(title='split'), title=c)
    fig.show()

0.03991287617714376 0.0403648275921001


0.05025429318919938 0.06071174417489411


0.05931693260458551 0.03721778749856566


0.05507892597451566 0.06305220916716185


In [ ]:
new

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(0, len(old)), y=old, name='Old Code Old Data'))
fig.add_trace(go.Scatter(x=np.arange(0, len(old)), y=old2, name='Old Code New Data'))
fig.add_trace(go.Scatter(x=np.arange(0, len(old)), y=new, name='New Code'))
fig.update_layout(yaxis=dict(title='RMSE'), xaxis=dict(title='split'))

In [ ]:
np.median(old2)

In [ ]:
res = []
country = []
model = []
for c in ['Mali', 'Nigeria', 'Syria','Yemen']:
  
    old = performances_forecasts(country=c, path='old_predictions/RC')
    model.append('old')
    res.append(old)  
    country.append(c)
      
        
    dfn = pd.read_csv(f"../FamPredAI/new_predictions/{c}_RC_100.csv")
    new = dfn.groupby('split').apply(lambda g: rmse(g.data, g.prediction)).values
    model.append('new')
    res.append(new)  
    country.append(c)
  

    hp = pd.read_csv(f"best_hyperparameters/HP_RC_{c}.csv")
    hp = hp["metric60"]
    
    model.append('hp')
    res.append(hp)  
    country.append(c)
    
    f = go.Figure()
    f.add_trace(go.Scatter(x=np.arange(len(old)), y=old, name='old'))
    f.add_trace(go.Scatter(x=np.arange(len(old)), y=new, name='new'))
    f.add_trace(go.Scatter(x=np.arange(len(old)), y=hp, name='cluster'))
    f.update_layout(title=c)
    f.show()
    
    
df = pd.DataFrame()
df['perf'] = res
df['model'] = model
df['country'] = country

In [ ]:
#### Compare Splits 

In [ ]:
res = []
country = []
model = []
for c in ['Mali', 'Nigeria', 'Syria','Yemen']:
  
    old = performances_forecasts(c)
    model.append('old')
    res.append(old)  
    country.append(c)
      
        
    dfn = pd.read_csv(f"../FamPredAI/new_predictions/{c}_RC_100.csv")
    new = dfn.groupby('split').apply(lambda g: rmse(g.data, g.prediction)).values
    model.append('new')
    res.append(new)  
    country.append(c)
  

    hp = pd.read_csv(f"grid_search_data/best_hyperparameters/HP_RC_{c}.csv")
    hp = hp["metric60"]
    
    model.append('hp')
    res.append(hp)  
    country.append(c)
    
    f = go.Figure()
    f.add_trace(go.Scatter(x=np.arange(len(old)), y=old, name='old'))
    f.add_trace(go.Scatter(x=np.arange(len(old)), y=new, name='new'))
    f.add_trace(go.Scatter(x=np.arange(len(old)), y=hp, name='cluster'))
    f.update_layout(title=c)
    f.show()
    
    
df = pd.DataFrame()
df['perf'] = res
df['model'] = model
df['country'] = country

In [ ]:
#### Compare Aggregates 

In [ ]:
res = []
country = []
model = []
for c in ['Mali', 'Nigeria', 'Syria','Yemen']:
  
    old = performances_forecasts(c)
    old=np.median(old)
    model.append('old')
    res.append(old)  
    country.append(c)
      
        
    dfn = pd.read_csv(f"../FamPredAI/new_predictions/{c}_RC_100.csv")
    new = dfn.groupby('split').apply(lambda g: rmse(g.data, g.prediction)).median()
    model.append('new')
    res.append(new)  
    country.append(c)
  

    hp = pd.read_csv(f"grid_search_data/best_hyperparameters/HP_RC_{c}.csv")
    hp = hp["metric60"].median()
    
    model.append('hp')
    res.append(hp)  
    country.append(c)
    
    
    
df = pd.DataFrame()
df['perf'] = res
df['model'] = model
df['country'] = country
px.bar(df, x='country', y='perf', color='model', barmode="group")

### Forecasts 

In [14]:
feature_dict = {"FCS": ["FCS"],
                "FCS+": ["FCS", "rCSI", "Ramadan", "day of the year", "rainfall_ndvi_seasonality"],
                "calendar": ["FCS", "Ramadan", "day of the year", "rainfall_ndvi_seasonality"],
                "climate": ["FCS", "rCSI", "Ramadan", "day of the year", "rainfall_ndvi_seasonality",
                             "rainfall", "NDVI", "log rainfall 1 month anomaly", "log rainfall 3 months anomaly",
                             "log NDVI anomaly"],
                'economics': ["FCS", "rCSI", "Ramadan", "day of the year",
                             "CE official", "CE unofficial","PEWI", "headline inflation", "food inflation"],
                "all":["FCS", "rCSI", "Ramadan", "day of the year", "rainfall_ndvi_seasonality",
                         "rainfall", "NDVI", "log rainfall 1 month anomaly", "log rainfall 3 months anomaly",
                         "log NDVI anomaly", "CE official", "CE unofficial","PEWI", "headline inflation",
                       "food inflation"]}


In [22]:
from datetime import datetime, timedelta

In [23]:
hp = pd.read_csv('best_hyperparameters/HP_RC_Mali.csv')
country='Mali'
row = hp.iloc[2,:]
hyperparameters = row[["w_in_scale",
                               "differencing",
                               "reg_param",
                               "n_rad",
                               "n_dim",
                               "features"]].to_dict()
hyperparameters['w_out_fit_flag'] = 'linear_and_square_r'
hyperparameters['train_sync_steps'] = 50
hyperparameters['n_avg_deg'] = 8
hyperparameters['smoothing'] = 10

constants_list = ["Ramadan", "day of the year", "lean season", "rainfall_ndvi_seasonality"]

In [37]:

target = 'FCS'
all_variables = pd.read_csv(f"data/{country}/full_timeseries_daily.csv", header=[0,1], index_col=[0])
all_variables = list(all_variables.melt().variable_0.unique())

variables = [v for v in feature_dict[hyperparameters['features']] if v in all_variables]
constants = [t for t in constants_list if t in variables]
variables = [v for v in variables if v not in constants_list]
if 'FCS' in variables:
    variables.remove('FCS')

In [38]:
first_forecast=datetime.strptime(row['split_date'], "%Y-%m-%d")
train_end_date = first_forecast - timedelta(days=1)

In [40]:
from model import Model

In [41]:
train_end_date = first_forecast - timedelta(days=1)
md = Model(country=country,
           forecasting_window=60,
           target_name=target,
           constants=constants,
           variable_names=variables,
           hyperparameters=hyperparameters
           )


#md.prepare_data()
#md.run(runs)

In [43]:
md.load_data_from_file(train_start_date=datetime(2017, 1, 1),
                       train_end_date=train_end_date)


In [52]:
md.prepare_data()

In [53]:
md.run(10)

In [57]:
px.line(md.predictions, x='date', y='prediction', color='adm1_code')

In [59]:
data = pd.read_csv('data/Mali/full_timeseries_daily.csv', header=[0,1], index_col=0)

In [ ]:
data

In [65]:
px.line(data.loc[datetime(2022, 8,1):datetime(2022,9,29), "FCS"])

TypeError: '<' not supported between instances of 'str' and 'datetime.datetime'

In [66]:
data

FCS                                                    \
                1926      1927      1928      1929      1930      1931   
2020-05-05  0.541197  0.586389  0.590994  0.291351  0.594128  0.544523   
2020-05-06  0.574372  0.624468  0.603889  0.292685  0.608250  0.564329   
2020-05-07  0.550794  0.613947  0.587809  0.292685  0.576694  0.548408   
2020-05-08  0.553401  0.621025  0.604454  0.301887  0.591252  0.518350   
2020-05-09  0.562498  0.601542  0.624241  0.316832  0.601512  0.526068   
...              ...       ...       ...       ...       ...       ...   
2024-01-22       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-23       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-24       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-25       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-26       NaN       NaN       NaN       NaN       NaN       NaN   

                                              rCSI  ... Remote violence  \
                1932      1933      1934      1926  ...            1927   
2020-05-05  0.468067  0.587341  0.664508  0.275415  ...       50.999983   
2020-05-06  0.473822  0.582408  0.657190  0.291867  ...       35.999919   
2020-05-07  0.481802  0.601157  0.657190  0.310063  ...       26.000057   
2020-05-08  0.442773  0.578871  0.657190  0.315688  ...       26.000073   
2020-05-09  0.398728  0.543297  0.654785  0.324071  ...       25.999892   
...              ...       ...       ...       ...  ...             ...   
2024-01-22       NaN       NaN       NaN       NaN  ...       -0.000213   
2024-01-23       NaN       NaN       NaN       NaN  ...       -0.000088   
2024-01-24       NaN       NaN       NaN       NaN  ...        0.000226   
2024-01-25       NaN       NaN       NaN       NaN  ...        0.000124   
2024-01-26       NaN       NaN       NaN       NaN  ...       -0.000066   

                                                                             \
                1928      1929      1930      1931       1932          1933   
2020-05-05 -0.000173 -0.000147  0.000275  7.000045  20.000013  1.098823e-04   
2020-05-06  0.000293  0.000034  0.000026  6.999840  19.999559  2.756315e-04   
2020-05-07  0.000312 -0.000031  0.000002  7.000063  20.000032  4.005348e-05   
2020-05-08 -0.000014 -0.000070 -0.000080  6.999936  20.000199  3.839800e-05   
2020-05-09 -0.000018  0.000037  0.000328  6.999774  20.000134  2.303301e-05   
...              ...       ...       ...       ...        ...           ...   
2024-01-22 -0.000095 -0.000106  0.000078  0.000107  -0.000026  6.728955e-06   
2024-01-23  0.000383  0.000160 -0.000368 -0.000155  -0.000133 -3.066696e-07   
2024-01-24  0.000153 -0.000258 -0.000150  0.000080  -0.000371  8.346679e-05   
2024-01-25 -0.000199 -0.000146  0.000002  0.000036   0.000020  3.457378e-04   
2024-01-26  0.000296  0.000327  0.000092  0.000077  -0.000285 -1.542113e-04   

                            Ramadan day of the year  
                 1934      national        national  
2020-05-05  45.999942  9.999989e-01      125.999486  
2020-05-06  46.000040  9.999976e-01      126.998288  
2020-05-07  32.000271  9.999990e-01      128.000100  
2020-05-08  31.999987  9.999976e-01      128.999279  
2020-05-09  31.999763  9.999996e-01      130.001340  
...               ...           ...             ...  
2024-01-22  -0.000081  2.547945e-06       21.999608  
2024-01-23   0.000025  2.606618e-07       23.001343  
2024-01-24  -0.000051  2.489839e-06       23.998901  
2024-01-25  -0.000229  3.865354e-06       24.998507  
2024-01-26  -0.000117 -5.970398e-07       25.998384  

[1362 rows x 113 columns]